In [1]:
import pandas as pd
import fastparquet


Período: 2015 a 2019
Casos moderados e graves (MG_QUALIFICADO = 1) se:
* SOROTERAPIA == Sim; ou
* SAESC/SAA > 0; ou
* COMPLICACOES_SISTEMICAS == Sim; ou
* EVOLUCAO == óbito por animais peçonhentos; ou
* MANIFESTACOES_SISTEMICAS == Sim e CLASSIFICACAO ≠ leve.

In [21]:
# Carrega banco completo
df = pd.read_parquet(path='Dados-processados/2_df_recodificado.parquet')

In [22]:
# Filtra intervalo de ano
df = df[df['ANO'].isin([2015,2019])]

# Filtra municípios ponderados
muni_ponde = ["AGUDOS","AREALVA" ,"AVAI" ,"IACANGA" ,"PEDERNEIRAS" ,"PIRATININGA" ,"CABREUVA" ,"CAMPO LIMPO PAULISTA" ,"ITUPEVA" ,"JARINU" ,"LOUVEIRA" ,"VARZEA PAULISTA" ,
"ALVARO DE CARVALHO" ,"ALVINLANDIA" ,"CAMPOS NOVOS PAULISTA" ,"ECHAPORA" ,"FERNAO" ,"GALIA" ,"GARCA" ,"GUAIMBE" ,"GUARANTA" ,"JULIO MESQUITA" ,"LUPERCIO" ,"OCAUCU" ,"ORIENTE" ,
"OSCAR BRESSANE" ,"POMPEIA" ,"QUINTANA" ,"UBIRAJARA" ,"VERA CRUZ" ]

dfp = df[df['NOME_MUNI'].isin(muni_ponde)]

In [32]:
print(
    df.groupby(['CS_SEXO']).size().reset_index(name='n'), '\n',
    dfp.groupby(['CS_SEXO']).size().reset_index(name='n')
)

     CS_SEXO      n
0   Feminino  21399
1  Masculino  27203 
      CS_SEXO    n
0   Feminino  226
1  Masculino  380


In [4]:
# converter ampolas para número
for col in ["NU_AMPOL_8", "NU_AMPOL_9"]:
    dfp[col] = pd.to_numeric(dfp[col], errors="coerce").fillna(0)

# =========================================================
# FILTRO DE CASOS MODERADOS/GRAVES QUALIFICADOS
# ESCORPIONISMO
# =========================================================

# ---------------------------------------------------------
# CRITÉRIOS FORTES
# Isoladamente já sugerem fortemente MG
# ---------------------------------------------------------

criterio_forte = (

    # SAA >= 2 ampolas
    (dfp["NU_AMPOL_8"] >= 2) |

    # SAEsc >= 2 ampolas
    (dfp["NU_AMPOL_9"] >= 2) |

    # Óbito por animais peçonhentos
    (dfp["EVOLUCAO"] == "Obito por ap") |

    # Manifestações vagais
    (dfp["CLI_VAGAIS"] == "Sim")

)

# ---------------------------------------------------------
# CRITÉRIOS ASSOCIATIVOS
# Variáveis sujeitas a erro de preenchimento,
# mas que em conjunto aumentam a probabilidade
# de representar MG
# ---------------------------------------------------------

criterio_associativo = (

    # Soroterapia + classificação moderado/grave
    (
        (dfp["CON_SOROTE"] == "Sim") &
        (dfp["TRA_CLASSI"].isin(["Moderado", "Grave"]))
    ) |

    # Soroterapia + manifestações sistêmicas
    (
        (dfp["CON_SOROTE"] == "Sim") &
        (dfp["MCLI_SIST"] == "Sim")
    ) |

    # Soroterapia + complicações sistêmicas
    (
        (dfp["CON_SOROTE"] == "Sim") &
        (dfp["COM_SISTEM"] == "Sim")
    )

)

# ---------------------------------------------------------
# FILTRO FINAL
# ---------------------------------------------------------

filtro_mg = criterio_forte | criterio_associativo

# ---------------------------------------------------------
# APLICAR FILTRO
# ---------------------------------------------------------

dfp_mg = dfp[filtro_mg].copy()

# ---------------------------------------------------------
# CRIAR VARIÁVEL BINÁRIA (OPCIONAL)
# ---------------------------------------------------------

dfp["MG_QUALIFICADO"] = filtro_mg.astype(object) # aceita texto e booleanos

# ---------------------------------------------------------
# CONFERÊNCIA
# ---------------------------------------------------------

print("Total de casos:", len(dfp))
print("Moderados/Graves qualificados:", filtro_mg.sum())
print("Proporção:", round(filtro_mg.mean() * 100, 2), "%")

Total de casos: 606
Moderados/Graves qualificados: 13
Proporção: 2.15 %


In [5]:
# Recodificando valores
dfp.loc[dfp['MG_QUALIFICADO'] == False, 'MG_QUALIFICADO'] = 'Leve'
dfp.loc[dfp['MG_QUALIFICADO'] == True, 'MG_QUALIFICADO'] = 'MG'

In [6]:
# Todos os casos sem restrição de idade (leves e moderados/graves)
total = pd.crosstab(dfp['NOME_MUNI'],dfp['MG_QUALIFICADO']).astype(int)
total


MG_QUALIFICADO,Leve,MG
NOME_MUNI,,
AGUDOS,13,0
ALVARO DE CARVALHO,6,0
ALVINLANDIA,2,0
AREALVA,57,2
AVAI,25,1
CABREUVA,10,0
CAMPO LIMPO PAULISTA,42,0
CAMPOS NOVOS PAULISTA,21,0
ECHAPORA,8,0


In [7]:
ate13 = pd.crosstab(dfp.loc[dfp['IDADE_ANOS'] <= 13,'NOME_MUNI'],dfp['MG_QUALIFICADO']).astype(int)
ate13

MG_QUALIFICADO,Leve,MG
NOME_MUNI,,
AGUDOS,3,0
ALVARO DE CARVALHO,1,0
AREALVA,10,1
AVAI,7,1
CABREUVA,1,0
CAMPO LIMPO PAULISTA,3,0
CAMPOS NOVOS PAULISTA,3,0
FERNAO,2,0
GARCA,7,0


In [8]:
# 1. Cria a tabela cruzada
ate14 = pd.crosstab(dfp.loc[dfp['IDADE_ANOS'] <= 14, 'NOME_MUNI'], dfp['MG_QUALIFICADO']).reset_index()

# 2. Limpa o nome do índice das colunas (boa prática para evitar problemas visuais)
ate14.columns.name = None

# 3. Cria a coluna de total somando as duas colunas com o operador +
# ATENÇÃO: Verifique se no seu DataFrame está 'Leve' ou 'leve', e 'MG' ou 'mg'.
ate14['total_ate14'] = ate14['Leve'] + ate14['MG']

ate14

,NOME_MUNI,Leve,MG,total_ate14
0,AGUDOS,3,0,3
1,ALVARO DE CARVALHO,1,0,1
2,AREALVA,11,1,12
3,AVAI,7,1,8
4,CABREUVA,1,0,1
5,CAMPO LIMPO PAULISTA,3,0,3
6,CAMPOS NOVOS PAULISTA,3,0,3
7,FERNAO,2,0,2
8,GARCA,10,0,10
9,GUAIMBE,1,0,1


In [9]:
ate17 = pd.crosstab(dfp.loc[dfp['IDADE_ANOS'] <= 17,'NOME_MUNI'],dfp['MG_QUALIFICADO'])
ate17.columns.name = None
ate17 = ate17.reset_index()
ate17.columns

Index(['NOME_MUNI', 'Leve', 'MG'], dtype='str')

In [10]:
# 1. Cria a tabela cruzada
ate17 = pd.crosstab(
    dfp.loc[dfp['IDADE_ANOS'] <= 17, 'NOME_MUNI'],
    dfp['MG_QUALIFICADO']
).fillna(0).astype(int)

# 2. Limpa o nome do índice das colunas e transforma em colunas normais
ate17.columns.name = None
ate17 = ate17.reset_index()

# 3. Seleciona APENAS as colunas 'NOME_MUNI' e a coluna 1
# Nota: Como o '1' original era um número inteiro, usamos 1 sem aspas. 
# Se no seu original ele for um texto, use '1' com aspas.
ate17_filtrado = ate17[['NOME_MUNI', 1]]

# Verificando o resultado
print(ate17_filtrado.columns)
ate17_filtrado[1]

In [43]:
dfp.loc[dfp['NOME_MUNI']=='OSCAR BRESSANE',['IDADE_ANOS']]


,IDADE_ANOS
164171,69
170863,60
171541,33
184405,25
189181,30
